# 06 - Feature Engineering Pix

Cria features temporais para EDA, classificação e regressão.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))


In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

from src.config import PIX_ML_FEATURES_CSV_DIR, PIX_ML_FEATURES_DIR, PIX_MONTHLY_INDICATORS_DIR, create_project_directories
from src.data_quality import ensure_not_empty
from src.spark_session import get_spark_session

create_project_directories(False)
spark = get_spark_session("06-feature-engineering-pix")

In [ ]:
monthly_df = spark.read.parquet(str(PIX_MONTHLY_INDICATORS_DIR)).orderBy("ano_mes")
ensure_not_empty(monthly_df, "Gold indicadores mensais")
w = Window.orderBy("ano_mes")
w3 = Window.orderBy("ano_mes").rowsBetween(-2, 0)

features_df = (
    monthly_df
    .withColumn("valor_total_lag_1", F.lag("valor_total").over(w))
    .withColumn("quantidade_transacoes_lag_1", F.lag("quantidade_transacoes").over(w))
    .withColumn("ticket_medio_lag_1", F.lag("ticket_medio").over(w))
    .withColumn("crescimento_valor_lag_1", F.lag("crescimento_valor_mes_anterior").over(w))
    .withColumn("crescimento_qtd_lag_1", F.lag("crescimento_qtd_mes_anterior").over(w))
    .withColumn("valor_total_mm3", F.avg("valor_total").over(w3))
    .withColumn("quantidade_transacoes_mm3", F.avg("quantidade_transacoes").over(w3))
    .withColumn("ticket_medio_mm3", F.avg("ticket_medio").over(w3))
    .withColumn("mes_numero", F.substring("ano_mes", 5, 2).cast("int"))
    .withColumn("trimestre", F.ceil(F.col("mes_numero") / F.lit(3)).cast("int"))
    .withColumn("flag_crescimento_valor", F.when(F.col("crescimento_valor_mes_anterior") > 0, F.lit(1)).otherwise(F.lit(0)))
    .withColumn("flag_crescimento_qtd", F.when(F.col("crescimento_qtd_mes_anterior") > 0, F.lit(1)).otherwise(F.lit(0)))
    .withColumn("classe_tendencia_valor", F.when(F.col("crescimento_valor_mes_anterior") > 1, "crescimento").when(F.col("crescimento_valor_mes_anterior") < -1, "queda").otherwise("estabilidade"))
    .withColumn("classe_tendencia_qtd", F.when(F.col("crescimento_qtd_mes_anterior") > 1, "crescimento").when(F.col("crescimento_qtd_mes_anterior") < -1, "queda").otherwise("estabilidade"))
    .fillna(0, subset=["valor_total_lag_1", "quantidade_transacoes_lag_1", "ticket_medio_lag_1", "crescimento_valor_lag_1", "crescimento_qtd_lag_1"])
)

features_df.write.mode("overwrite").parquet(str(PIX_ML_FEATURES_DIR))
features_df.coalesce(1).write.mode("overwrite").option("header", True).csv(str(PIX_ML_FEATURES_CSV_DIR))
features_df.show(20, truncate=False)

In [ ]:
spark.stop()